# LeetCode 884 — Uncommon Words from Two Sentences bằng Python

**Link đề:** https://leetcode.com/problems/uncommon-words-from-two-sentences/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Counting

```python
class Solution:
    def uncommonFromSentences(self, s1: str, s2: str) -> list[str]:
        ...
```

## Mục tiêu học
1. Tách câu thành từ bằng `str.split()`, nối list bằng `+`.
2. Nắm **đếm tần suất** bằng `dict` thủ công, `dict.get`, và `collections.Counter`.
3. Đi từ cách đếm thô `list.count` (`O(W²)`) tới cách hash map `O(W)`.
4. Phát hiện **insight then chốt**: "xuất hiện đúng 1 lần ở một câu và không xuất hiện ở câu kia" ⇔ "xuất hiện **đúng 1 lần** trong **hai câu gộp lại**".
5. Trace, chứng minh, test đối chiếu (so sánh kết quả **không phụ thuộc thứ tự**) và 6 bài tập.

## 1. Đề bài và ví dụ

Một **câu** là chuỗi các **từ** ngăn bởi **đúng một dấu cách**, mỗi từ gồm chữ thường.

Một từ là **uncommon** (không chung) nếu nó xuất hiện **đúng một lần** trong một câu và **không xuất hiện** trong câu còn lại.

Cho `s1`, `s2`, trả về list các từ uncommon (thứ tự **tùy ý**).

| s1 | s2 | Output |
|---|---|---|
| `"this apple is sweet"` | `"this apple is sour"` | `["sweet","sour"]` |
| `"apple apple"` | `"banana"` | `["banana"]` |

Ví dụ 2: `apple` xuất hiện **2 lần** trong `s1` → không phải "đúng một lần" → loại.

**Constraints:** `1 <= s1.length, s2.length <= 200`; chỉ chữ thường và dấu cách; không có dấu cách thừa ở đầu/cuối; các từ cách nhau đúng 1 dấu cách.

### 1.1. Phân loại từng từ của ví dụ 1 bằng tay

| Từ | Số lần trong s1 | Số lần trong s2 | Uncommon? |
|---|---:|---:|:---:|
| this | 1 | 1 | không (có ở cả hai) |
| apple | 1 | 1 | không |
| is | 1 | 1 | không |
| sweet | 1 | 0 | **có** |
| sour | 0 | 1 | **có** |

Quan sát cột cuối: từ uncommon có **tổng số lần = 1**. Giữ ý này lại — nó là chìa khóa ở mục 5.

## 2. Nền tảng Python cần biết

### 2.1. `str.split()` — tách câu thành list từ

`"a b c".split()` → `["a", "b", "c"]`. Không truyền đối số thì Python tách theo **mọi khoảng trắng** và bỏ chuỗi rỗng. `split(" ")` thì tách đúng theo một dấu cách (với đề này hai cách cho cùng kết quả vì không có dấu cách thừa).

In [1]:
s1 = "this apple is sweet"
print(s1.split())
print("a  b".split(), "vs", "a  b".split(" "), "<- khác nhau khi có 2 dấu cách liền")

['this', 'apple', 'is', 'sweet']
['a', 'b'] vs ['a', '', 'b'] <- khác nhau khi có 2 dấu cách liền


### 2.2. Nối hai list bằng `+`

`[1, 2] + [3]` → `[1, 2, 3]` (tạo list **mới**). Ta sẽ dùng để gộp từ của hai câu: `s1.split() + s2.split()`.

In [2]:
words = "this apple is sweet".split() + "this apple is sour".split()
print(words, "| tổng số từ:", len(words))

['this', 'apple', 'is', 'sweet', 'this', 'apple', 'is', 'sour'] | tổng số từ: 8


### 2.3. Đếm tần suất bằng `dict`

`dict` ánh xạ **key → value**. Ở đây key = từ, value = số lần xuất hiện.

- `d.get(k, 0)`: lấy `d[k]` nếu có, nếu không trả `0` (không lỗi `KeyError`).
- Mẫu đếm kinh điển: `d[w] = d.get(w, 0) + 1`.

In [3]:
freq = {}
for w in ["apple", "banana", "apple"]:
    freq[w] = freq.get(w, 0) + 1
    print(f"gặp {w!r:9} -> freq = {freq}")

try:
    freq["cherry"]
except KeyError as e:
    print("freq['cherry'] lỗi KeyError:", e, "| freq.get('cherry', 0) =", freq.get("cherry", 0))

gặp 'apple'   -> freq = {'apple': 1}
gặp 'banana'  -> freq = {'apple': 1, 'banana': 1}
gặp 'apple'   -> freq = {'apple': 2, 'banana': 1}
freq['cherry'] lỗi KeyError: 'cherry' | freq.get('cherry', 0) = 0


### 2.4. `collections.Counter` — dict chuyên để đếm

`Counter(iterable)` đếm sẵn mọi phần tử. Truy cập key không tồn tại trả `0` (không lỗi). Hai Counter có thể **cộng** nhau: `Counter(a) + Counter(b)`.

In [4]:
from collections import Counter

c1 = Counter("apple apple".split())
c2 = Counter("banana".split())
print("c1 =", c1, "| c1['banana'] =", c1["banana"])
print("c1 + c2 =", c1 + c2)
print("items():", list((c1 + c2).items()))

c1 = Counter({'apple': 2}) | c1['banana'] = 0
c1 + c2 = Counter({'apple': 2, 'banana': 1})
items(): [('apple', 2), ('banana', 1)]


### 2.5. List comprehension có điều kiện

`[w for w, c in cnt.items() if c == 1]` đọc là: "lấy `w` cho mỗi cặp `(w, c)` trong `cnt.items()` **nếu** `c == 1`". Đây là cách lọc gọn trong một dòng.

In [5]:
cnt = Counter(["a", "b", "a", "c"])
print([w for w, c in cnt.items() if c == 1])

['b', 'c']


## 3. Cách 1 — Baseline: dùng `list.count` cho từng từ

Làm đúng nguyên văn định nghĩa:
- Với mỗi từ `w` trong `s1`: nếu `words1.count(w) == 1` và `words2.count(w) == 0` → nhận.
- Làm tương tự cho `s2`.

`list.count(x)` duyệt **toàn bộ** list để đếm → `O(len)` mỗi lần gọi.

In [6]:
def uncommon_bruteforce(s1: str, s2: str) -> list[str]:
    words1, words2 = s1.split(), s2.split()
    result = []
    for w in words1:
        if words1.count(w) == 1 and words2.count(w) == 0:
            result.append(w)
    for w in words2:
        if words2.count(w) == 1 and words1.count(w) == 0:
            result.append(w)
    return result

print(uncommon_bruteforce("this apple is sweet", "this apple is sour"))
print(uncommon_bruteforce("apple apple", "banana"))

['sweet', 'sour']
['banana']


### 3.1. Nút thắt

Gọi `W` = tổng số từ. Mỗi từ gọi `count` trên cả hai list → `O(W)` mỗi từ → tổng `O(W²)`.

Với đề (≤ 200 ký tự, tức ≤ ~100 từ) vẫn nhanh. Nhưng nếu câu dài 10^5 từ thì `10^10` phép → quá chậm.

**Vấn đề:** ta **đếm lại** cùng một thông tin nhiều lần. **Insight 1:** đếm **một lần** cho mọi từ, lưu vào hash map, sau đó tra cứu `O(1)` trung bình.

## 4. Cách 2 — Hai bảng đếm, kiểm tra đúng định nghĩa

Đếm riêng `cnt1` (cho s1) và `cnt2` (cho s2). Một từ uncommon nếu:
- `cnt1[w] == 1 and cnt2[w] == 0`, **hoặc**
- `cnt2[w] == 1 and cnt1[w] == 0`.

In [7]:
def uncommon_two_counters(s1: str, s2: str) -> list[str]:
    cnt1, cnt2 = Counter(s1.split()), Counter(s2.split())
    result = []
    for w in cnt1:
        if cnt1[w] == 1 and cnt2[w] == 0:
            result.append(w)
    for w in cnt2:
        if cnt2[w] == 1 and cnt1[w] == 0:
            result.append(w)
    return result

print(uncommon_two_counters("this apple is sweet", "this apple is sour"))
print(uncommon_two_counters("apple apple", "banana"))

['sweet', 'sour']
['banana']


Duyệt `for w in cnt1` đi qua các **key khác nhau** — mỗi từ chỉ xét một lần, không bị thêm trùng vào kết quả.

## 5. Insight chính: gộp hai câu, lấy từ có tổng số lần bằng 1

Gọi `a = cnt1[w]`, `b = cnt2[w]` (số nguyên ≥ 0). Điều kiện uncommon:

```text
(a == 1 and b == 0)  or  (a == 0 and b == 1)
```

Xét `a + b == 1` với `a, b ≥ 0` nguyên: chỉ có hai khả năng `(1, 0)` hoặc `(0, 1)` — **trùng khớp hoàn toàn** điều kiện trên!

Vậy: **đếm tất cả từ của cả hai câu chung một bảng**, lấy các từ có số lần **đúng bằng 1**.

In [8]:
# Kiểm tra tương đương bằng cách liệt kê mọi (a, b) nhỏ
for a in range(3):
    for b_ in range(3):
        cond_def = (a == 1 and b_ == 0) or (a == 0 and b_ == 1)
        cond_sum = (a + b_ == 1)
        assert cond_def == cond_sum
        print(f"a={a}, b={b_}: định nghĩa={cond_def!s:5} | a+b==1: {cond_sum}")

a=0, b=0: định nghĩa=False | a+b==1: False
a=0, b=1: định nghĩa=True  | a+b==1: True
a=0, b=2: định nghĩa=False | a+b==1: False
a=1, b=0: định nghĩa=True  | a+b==1: True
a=1, b=1: định nghĩa=False | a+b==1: False
a=1, b=2: định nghĩa=False | a+b==1: False
a=2, b=0: định nghĩa=False | a+b==1: False
a=2, b=1: định nghĩa=False | a+b==1: False
a=2, b=2: định nghĩa=False | a+b==1: False


## 6. Cách 3 — Lời giải chính: một `Counter` cho câu gộp

Ghép mảnh:
1. `words = s1.split() + s2.split()` (hoặc `(s1 + " " + s2).split()`).
2. `cnt = Counter(words)`.
3. Trả `[w for w, c in cnt.items() if c == 1]`.

In [9]:
def uncommon_counter(s1: str, s2: str) -> list[str]:
    cnt = Counter(s1.split() + s2.split())
    return [w for w, c in cnt.items() if c == 1]

print(uncommon_counter("this apple is sweet", "this apple is sour"))
print(uncommon_counter("apple apple", "banana"))

['sweet', 'sour']
['banana']


### 6.1. Cùng ý tưởng nhưng tự viết `dict` (không dùng thư viện)

Phỏng vấn đôi khi yêu cầu tự đếm. Ta dùng mẫu `freq[w] = freq.get(w, 0) + 1`.

In [10]:
def uncommon_dict(s1: str, s2: str) -> list[str]:
    freq: dict[str, int] = {}
    for w in s1.split():
        freq[w] = freq.get(w, 0) + 1
    for w in s2.split():
        freq[w] = freq.get(w, 0) + 1
    return [w for w in freq if freq[w] == 1]

print(uncommon_dict("this apple is sweet", "this apple is sour"))

['sweet', 'sour']


### 6.2. Cách 4 — Hai `set`: "đã gặp" và "gặp nhiều lần"

Không cần đếm chính xác, chỉ cần phân biệt "1 lần" và "≥ 2 lần":
- `seen`: các từ đã gặp ít nhất 1 lần.
- `dup`: các từ đã gặp ít nhất 2 lần.

Đáp án = `seen - dup` (hiệu tập hợp: phần tử có trong `seen` nhưng không có trong `dup`).

In [11]:
print({1, 2, 3} - {2}, "<- phép hiệu tập hợp")

def uncommon_sets(s1: str, s2: str) -> list[str]:
    seen, dup = set(), set()
    for w in (s1 + " " + s2).split():
        if w in seen:
            dup.add(w)
        else:
            seen.add(w)
    return list(seen - dup)

print(sorted(uncommon_sets("this apple is sweet", "this apple is sour")))

{1, 3} <- phép hiệu tập hợp
['sour', 'sweet']


Ghi chú: `set` không giữ thứ tự chèn, nên kết quả cách 4 có thể ra thứ tự khác lần chạy khác. Đề chấp nhận **mọi thứ tự**, nhưng khi in/test ta nên `sorted(...)` cho ổn định.

## 7. TRACE từng bước

### 7.1. Bảng trace cách `dict` với `s1 = "this apple is sweet"`, `s2 = "this apple is sour"`

| Bước | Từ | freq trước | freq sau |
|---:|---|---|---|
| 1 | this | {} | {this:1} |
| 2 | apple | {this:1} | {this:1, apple:1} |
| 3 | is | … | {…, is:1} |
| 4 | sweet | … | {…, sweet:1} |
| 5 | this | this:1 | this:**2** |
| 6 | apple | apple:1 | apple:**2** |
| 7 | is | is:1 | is:**2** |
| 8 | sour | — | sour:1 |

Lọc `freq == 1` → `sweet`, `sour`.

In [12]:
def trace_uncommon(s1: str, s2: str) -> list[str]:
    words = s1.split() + s2.split()
    print("Danh sách từ gộp:", words)
    freq = {}
    for step, w in enumerate(words, start=1):
        before = freq.get(w, 0)
        freq[w] = before + 1
        print(f"Bước {step}: '{w}' {before} -> {freq[w]}")
    print("Bảng tần suất cuối:", freq)
    result = []
    for w, c in freq.items():
        decision = "GIỮ" if c == 1 else "loại"
        print(f"  {w!r:8} xuất hiện {c} lần -> {decision}")
        if c == 1:
            result.append(w)
    print("Kết quả:", result)
    return result

_ = trace_uncommon("this apple is sweet", "this apple is sour")

Danh sách từ gộp: ['this', 'apple', 'is', 'sweet', 'this', 'apple', 'is', 'sour']
Bước 1: 'this' 0 -> 1
Bước 2: 'apple' 0 -> 1
Bước 3: 'is' 0 -> 1
Bước 4: 'sweet' 0 -> 1
Bước 5: 'this' 1 -> 2
Bước 6: 'apple' 1 -> 2
Bước 7: 'is' 1 -> 2
Bước 8: 'sour' 0 -> 1
Bảng tần suất cuối: {'this': 2, 'apple': 2, 'is': 2, 'sweet': 1, 'sour': 1}
  'this'   xuất hiện 2 lần -> loại
  'apple'  xuất hiện 2 lần -> loại
  'is'     xuất hiện 2 lần -> loại
  'sweet'  xuất hiện 1 lần -> GIỮ
  'sour'   xuất hiện 1 lần -> GIỮ
Kết quả: ['sweet', 'sour']


`enumerate(words, start=1)` đánh số từ 1 thay vì 0 — tiện cho in "Bước 1, 2, ...".

### 7.2. Trace ví dụ có từ lặp trong cùng một câu

In [13]:
_ = trace_uncommon("apple apple", "banana")

Danh sách từ gộp: ['apple', 'apple', 'banana']
Bước 1: 'apple' 0 -> 1
Bước 2: 'apple' 1 -> 2
Bước 3: 'banana' 0 -> 1
Bảng tần suất cuối: {'apple': 2, 'banana': 1}
  'apple'  xuất hiện 2 lần -> loại
  'banana' xuất hiện 1 lần -> GIỮ
Kết quả: ['banana']


### 7.3. Trace một ca "bẫy": từ chỉ xuất hiện 1 lần ở mỗi câu

`s1 = "a b"`, `s2 = "b c"`: `b` xuất hiện 1 lần trong s1 và 1 lần trong s2 → tổng 2 → **loại** (vì nó có trong câu kia). Kết quả `["a", "c"]`.

In [14]:
_ = trace_uncommon("a b", "b c")

Danh sách từ gộp: ['a', 'b', 'b', 'c']
Bước 1: 'a' 0 -> 1
Bước 2: 'b' 0 -> 1
Bước 3: 'b' 1 -> 2
Bước 4: 'c' 0 -> 1
Bảng tần suất cuối: {'a': 1, 'b': 2, 'c': 1}
  'a'      xuất hiện 1 lần -> GIỮ
  'b'      xuất hiện 2 lần -> loại
  'c'      xuất hiện 1 lần -> GIỮ
Kết quả: ['a', 'c']


## 8. Code nộp LeetCode

In [15]:
from collections import Counter

class Solution:
    def uncommonFromSentences(self, s1: str, s2: str) -> list[str]:
        cnt = Counter(s1.split() + s2.split())
        return [w for w, c in cnt.items() if c == 1]

sol = Solution()
print(sol.uncommonFromSentences("this apple is sweet", "this apple is sour"))
print(sol.uncommonFromSentences("apple apple", "banana"))

['sweet', 'sour']
['banana']


### 8.1. Bản nộp không dùng `Counter`

In [16]:
class SolutionDict:
    def uncommonFromSentences(self, s1: str, s2: str) -> list[str]:
        freq = {}
        for w in (s1 + " " + s2).split():
            freq[w] = freq.get(w, 0) + 1
        return [w for w, c in freq.items() if c == 1]

print(SolutionDict().uncommonFromSentences("a b", "b c"))

['a', 'c']


## 9. Vì sao đúng?

1. **Đếm đúng:** sau vòng đếm, `freq[w] = cnt1[w] + cnt2[w]` với mọi từ `w` (mỗi lần xuất hiện cộng đúng 1).
   *Invariant:* sau khi xử lý `k` từ đầu của danh sách gộp, `freq[w]` = số lần `w` xuất hiện trong `k` từ đó.
2. **Lọc đúng:** đã chứng minh ở mục 5: `cnt1 + cnt2 == 1` ⇔ `(1,0)` hoặc `(0,1)` ⇔ định nghĩa uncommon.
3. **Không trùng lặp kết quả:** duyệt key của dict, mỗi từ khác nhau xuất hiện đúng một lần.
4. **Không bỏ sót:** mọi từ của hai câu đều có trong `freq` nên đều được xét.

## 10. Độ phức tạp và so sánh

Gọi `L = len(s1) + len(s2)` (số ký tự), `W` = tổng số từ.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. `list.count` | đếm lại cho từng từ | `O(W · L)` ≈ `O(W²)` từ | `O(L)` | Dễ hiểu, chậm khi lớn |
| 2. Hai Counter | đúng định nghĩa | `O(L)` | `O(L)` | Rõ ràng |
| 3. Một Counter | tổng = 1 | `O(L)` | `O(L)` | **Gọn nhất** |
| 4. Hai set | seen / dup | `O(L)` | `O(L)` | Không cần số đếm chính xác |

Thời gian `O(L)` vì `split` duyệt mọi ký tự, và mỗi lần băm (hash) một từ tốn thời gian tỉ lệ độ dài từ. Tra cứu dict/set là `O(1)` **trung bình** (trường hợp xấu có thể chậm hơn do va chạm hash).

## 11. Bộ test: mẫu đề, biên, đối chiếu

Vì đề cho phép **thứ tự tùy ý**, ta so sánh `sorted(kết quả)` với `sorted(expected)`.

In [17]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS  {name}: {sorted(actual)}")

methods = [uncommon_bruteforce, uncommon_two_counters, uncommon_counter, uncommon_dict, uncommon_sets,
           Solution().uncommonFromSentences, SolutionDict().uncommonFromSentences]

cases = [
    ("đề VD1", "this apple is sweet", "this apple is sour", ["sweet", "sour"]),
    ("đề VD2", "apple apple", "banana", ["banana"]),
    ("giống hệt", "a b c", "a b c", []),
    ("không chung từ nào", "x y", "z", ["x", "y", "z"]),
    ("lặp trong cùng câu", "a a b", "c c", ["b"]),
    ("một từ mỗi câu, trùng", "z", "z", []),
    ("một từ mỗi câu, khác", "z", "q", ["z", "q"]),
    ("từ là tiền tố của nhau", "app apple", "apple", ["app"]),
]
for name, a, c, exp in cases:
    for f in methods:
        assert sorted(f(a, c)) == sorted(exp), (name, f.__name__)
    check(name, uncommon_counter(a, c), exp)

PASS  đề VD1: ['sour', 'sweet']
PASS  đề VD2: ['banana']
PASS  giống hệt: []
PASS  không chung từ nào: ['x', 'y', 'z']
PASS  lặp trong cùng câu: ['b']
PASS  một từ mỗi câu, trùng: []
PASS  một từ mỗi câu, khác: ['q', 'z']
PASS  từ là tiền tố của nhau: ['app']


In [18]:
import random
random.seed(884)
vocab = ["a", "b", "c", "d", "e", "ab"]
for _ in range(500):
    s1 = " ".join(random.choice(vocab) for _ in range(random.randint(1, 8)))
    s2 = " ".join(random.choice(vocab) for _ in range(random.randint(1, 8)))
    expected = sorted(uncommon_bruteforce(s1, s2))
    for f in methods:
        assert sorted(f(s1, s2)) == expected
print("500 cặp câu ngẫu nhiên: 7 cách luôn cho cùng tập kết quả.")

500 cặp câu ngẫu nhiên: 7 cách luôn cho cùng tập kết quả.


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng phép đối xứng tập hợp `set1 ^ set2`
`^` cho từ có trong **đúng một** tập, nhưng **bỏ qua số lần lặp** trong cùng câu → `"apple apple"` vs `"banana"` sẽ ra cả `apple`.

In [19]:
def wrong_symmetric_difference(s1, s2):
    return list(set(s1.split()) ^ set(s2.split()))

print("Sai :", sorted(wrong_symmetric_difference("apple apple", "banana")))
print("Đúng:", sorted(uncommon_counter("apple apple", "banana")))

Sai : ['apple', 'banana']
Đúng: ['banana']


### Lỗi 2: Chỉ kiểm "xuất hiện 1 lần trong câu của nó", quên kiểm câu kia
Với `"a b"` và `"b c"`, `b` xuất hiện 1 lần trong mỗi câu → bị nhận nhầm (và còn bị thêm **hai lần**).

In [20]:
def wrong_only_own_sentence(s1, s2):
    w1, w2 = s1.split(), s2.split()
    return [w for w in w1 if w1.count(w) == 1] + [w for w in w2 if w2.count(w) == 1]

print("Sai :", wrong_only_own_sentence("a b", "b c"))
print("Đúng:", uncommon_counter("a b", "b c"))

Sai : ['a', 'b', 'b', 'c']
Đúng: ['a', 'c']


### Lỗi 3: Duyệt list từ (có lặp) thay vì key của bảng đếm, khi điều kiện là `>= 1`
Nếu bài biến thể chấp nhận "xuất hiện ≥ 1 lần" mà ta duyệt list từ, từ lặp sẽ bị thêm nhiều lần. Duyệt **key** của dict giúp mỗi từ chỉ xét một lần.

### Lỗi 4: Nối hai câu thiếu dấu cách `s1 + s2`
`"ab" + "cd"` → `"abcd"`: từ cuối của s1 dính vào từ đầu của s2!

In [21]:
print("Thiếu dấu cách:", ("this apple" + "is sour").split())
print("Đúng          :", ("this apple" + " " + "is sour").split())

Thiếu dấu cách: ['this', 'appleis', 'sour']
Đúng          : ['this', 'apple', 'is', 'sour']


### Lỗi 5: So sánh kết quả với expected bằng `==` khi đề cho phép thứ tự tùy ý
Code đúng nhưng test fail vì thứ tự khác. Luôn so sánh `sorted(...)` hoặc `set(...)` (khi không có trùng lặp).

## 13. Mẹo nhận dạng pattern

- Đề có "xuất hiện **đúng k lần**", "duy nhất", "không lặp", "tần suất" → **bảng đếm** (`dict`/`Counter`).
- Điều kiện trên **hai** nguồn dữ liệu mà chỉ phụ thuộc **tổng số lần** → **gộp** hai nguồn rồi đếm một lần.
- Chỉ cần phân biệt 0 / 1 / ≥2 lần → hai `set` (`seen`, `dup`) là đủ.
- Kết quả "thứ tự tùy ý" → được dùng `set`, nhưng test bằng `sorted`.

Bài cùng họ: 387 First Unique Character, 1207 Unique Number of Occurrences, 2085 Count Common Words With One Occurrence, 349 Intersection of Two Arrays.

## 14. Biến thể

### 14.1. LeetCode 2085: đếm từ xuất hiện **đúng 1 lần ở mỗi** câu
Lúc này **không** gộp được (vì `(1,1)` khác `(2,0)` dù tổng đều là 2) → cần hai Counter.

In [22]:
def count_words_once_in_both(words1: list[str], words2: list[str]) -> int:
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w in c1 if c1[w] == 1 and c2[w] == 1)

print(count_words_once_in_both(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))   # 2

2


### 14.2. Uncommon cho **k câu**
"Xuất hiện đúng 1 lần trong một câu và không có ở các câu khác" ⇔ tổng số lần trên k câu bằng 1. Insight gộp vẫn dùng được!

In [23]:
def uncommon_k(sentences: list[str]) -> list[str]:
    cnt = Counter(w for s in sentences for w in s.split())
    return [w for w, c in cnt.items() if c == 1]

print(sorted(uncommon_k(["a b", "b c", "d a e"])))

['c', 'd', 'e']


### 14.3. Giữ nguyên thứ tự xuất hiện
`Counter`/`dict` trong Python 3.7+ **giữ thứ tự chèn**, nên cách 3 trả theo thứ tự lần đầu gặp. Cách `set` thì không.

## 15. Bài tập tự luyện

Các hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
`s1 = "the cat and the hat"`, `s2 = "a cat in a hat"`. Lập bảng tần suất gộp và chỉ ra các từ uncommon.

### Bài 2 — Dự đoán output
```python
Counter("b a b".split()) + Counter("a c".split())
"x  y".split(" ")
```

### Bài 3 — Điền chỗ trống: đếm bằng dict

In [24]:
def word_freq(s: str) -> dict[str, int]:
    freq = {}
    for w in s.split():
        # TODO: tăng freq[w] lên 1 (dùng get)
        pass
    return freq

# assert word_freq("a b a") == {"a": 2, "b": 1}

### Bài 4 — Tự viết `uncommonFromSentences` bằng hai set

In [25]:
def uncommon_exercise(s1: str, s2: str) -> list[str]:
    # TODO: seen, dup; trả seen - dup
    pass

# assert sorted(uncommon_exercise("this apple is sweet", "this apple is sour")) == ["sour", "sweet"]
# assert uncommon_exercise("apple apple", "banana") == ["banana"]

### Bài 5 — Biến thể: từ xuất hiện ở **cả hai** câu (mỗi từ trả 1 lần, sắp xếp tăng dần)

In [26]:
def common_words(s1: str, s2: str) -> list[str]:
    # TODO: dùng phép giao tập hợp &
    pass

# assert common_words("this apple is sweet", "this apple is sour") == ["apple", "is", "this"]

### Bài 6 — Mini challenge: từ xuất hiện đúng `k` lần tính trên cả hai câu, trả theo thứ tự xuất hiện đầu tiên

In [27]:
def words_exactly_k(s1: str, s2: str, k: int) -> list[str]:
    # TODO
    pass

# assert words_exactly_k("a b a", "c a b", 2) == ["b"]
# assert words_exactly_k("a b a", "c a b", 3) == ["a"]
# assert words_exactly_k("a b a", "c a b", 1) == ["c"]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Gộp hai câu | the:2, cat:2, hat:2, a:2 | còn lại `and`, `in` |
| 2 | Counter cộng theo key | `split(" ")` giữ chuỗi rỗng | `{'b':2,'a':2,'c':1}`, `['x','','y']` |
| 3 | mẫu đếm | `freq.get(w, 0)` | `freq[w] = freq.get(w, 0) + 1` |
| 4 | đã gặp → dup | `if w in seen` | `return list(seen - dup)` |
| 5 | giao hai tập | `set(a) & set(b)` | `sorted(...)` |
| 6 | Counter giữ thứ tự chèn | lọc `c == k` | list comprehension trên `items()` |

## 17. Đáp án tham khảo

**Bài 1:** gộp: `the:2, cat:2, and:1, hat:2, a:2, in:1` → uncommon: `["and", "in"]`.

**Bài 2:** `Counter({'b': 2, 'a': 2, 'c': 1})` và `['x', '', 'y']`.

In [28]:
def ans_word_freq(s):
    freq = {}
    for w in s.split():
        freq[w] = freq.get(w, 0) + 1
    return freq

def ans_uncommon_sets(s1, s2):
    seen, dup = set(), set()
    for w in (s1 + " " + s2).split():
        if w in seen:
            dup.add(w)
        else:
            seen.add(w)
    return list(seen - dup)

def ans_common_words(s1, s2):
    return sorted(set(s1.split()) & set(s2.split()))

def ans_words_exactly_k(s1, s2, k):
    cnt = Counter(s1.split() + s2.split())
    return [w for w, c in cnt.items() if c == k]

assert sorted(uncommon_counter("the cat and the hat", "a cat in a hat")) == ["and", "in"]
assert Counter("b a b".split()) + Counter("a c".split()) == Counter({"b": 2, "a": 2, "c": 1})
assert "x  y".split(" ") == ["x", "", "y"]
assert ans_word_freq("a b a") == {"a": 2, "b": 1}
assert sorted(ans_uncommon_sets("this apple is sweet", "this apple is sour")) == ["sour", "sweet"]
assert ans_uncommon_sets("apple apple", "banana") == ["banana"]
assert ans_common_words("this apple is sweet", "this apple is sour") == ["apple", "is", "this"]
assert ans_words_exactly_k("a b a", "c a b", 2) == ["b"]
assert ans_words_exactly_k("a b a", "c a b", 3) == ["a"]
assert ans_words_exactly_k("a b a", "c a b", 1) == ["c"]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
from collections import Counter

class Solution:
    def uncommonFromSentences(self, s1: str, s2: str) -> list[str]:
        cnt = Counter(s1.split() + s2.split())       # gộp hai câu, đếm một lần
        return [w for w, c in cnt.items() if c == 1] # tổng đúng 1 lần = uncommon
```

**Câu nhớ:** *"Uncommon = xuất hiện đúng một lần trong cả hai câu gộp lại."*

**Checklist:**
- [ ] Dùng **số đếm** (không chỉ `set`), vì từ lặp trong cùng câu phải bị loại?
- [ ] Gộp đúng: `s1.split() + s2.split()` hoặc nối có dấu cách?
- [ ] Điều kiện `== 1` (không phải `<= 1` hay `>= 1`)?
- [ ] Test có từ lặp trong một câu và từ chung giữa hai câu?
- [ ] Khi tự test, so sánh `sorted(...)` vì thứ tự tùy ý?